In [71]:
import pandas as pd
import os
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
import string
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)

nlp = spacy.load("en_core_web_lg")

In [72]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)  # This shows full column content
pd.set_option('display.width', None)  # Auto-detect display width
pd.set_option('display.max_seq_items', None)  # Show all items in lists

In [73]:
df = pd.read_csv('../data/dcInbox/dcinbox_export_114.csv')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()
df = df[pd.to_numeric(df['Unix Timestamp'], errors='coerce').notna()].copy()
df['datetime'] = pd.to_datetime(df['Unix Timestamp'], unit='ms')

# Sort chronologically
df = df.sort_values('datetime').reset_index(drop=True)

# Filter data for February and March
# df = df[df['datetime'].between('2016-01-01', '2016-12-31')]
df = df[df['Chamber'] == 'House']
republican_emails = df[df['Party'] == 'Republican']
democrat_emails = df[df['Party'] == 'Democrat']

/var/folders/02/c1hvrmj11kx0z457p84l6pbc0000gn/T/ipykernel_31509/2968176641.py:1: DtypeWarning: Columns (2,4,10,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,25

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20582 entries, 3 to 24132
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Subject         20581 non-null  object        
 1   Body            20582 non-null  object        
 2   Unix Timestamp  20582 non-null  object        
 3   BioGuide ID     20582 non-null  object        
 4   Congress        20582 non-null  object        
 5   First Name      20582 non-null  object        
 6   Last Name       20582 non-null  object        
 7   Date of Birth   20582 non-null  object        
 8   Gender          20582 non-null  object        
 9   State           20582 non-null  object        
 10  District        20582 non-null  object        
 11  Party           20582 non-null  object        
 12  Chamber         20582 non-null  object        
 13  Nickname        1084 non-null   object        
 14  ID              20582 non-null  object        
 15  datetim

In [75]:
## Preprocessing functions to apply to catalogs' aggregated course descriptions
def remove_double_spaces(text):
    if not text:
        return ""
    return " ".join(text.split())


def remove_paragraphs_dash(text):
    return re.sub(r"-\n", "", text)


def remove_paragraphs(text):
    return re.sub("\n", " ", text)

In [76]:
THEMATIC_LEXICON = {
    # Shared/Bipartisan Categories
    'Healthcare': [
        # From both parties
        'health care',
        'healthcare',
        'obamacare',
        'affordable care act',
        'medicare',
        'medicaid',
        'public health',
        'mental health',
        'health',  # Dem-leaning
        'repeal',  # Rep-leaning (context: "repeal Obamacare")
        'repeal and replace',  # Rep-leaning
        'affordable care', # dem term
    ],
    
    'Education': [
        'education',
        'student',  # Dem focus
        'college',  # Dem focus
        'student loan',  # Dem focus
        'community college',  # Dem focus
        'school choice',  # Rep focus
        'parental rights',  # Rep focus
        'curriculum',
        'crt',  # Rep focus
        'critical race theory',  # Rep focus
        'higher education',  # Dem focus
    ],
    
    'Gun Policy': [
        'gun violence',  # Dem frame
        'gun safety',  # Dem frame
        'gun control',  # Both parties
        'gun reform',  # Dem frame
        'gun',
        'violence',
        'second amendment',  # Rep frame
        '2nd amendment',  # Rep frame
        'gun rights',  # Rep frame
        'right to bear arms',  # Rep frame
        'gun violence'
    ],
    
    # for presentation
    'Immigration': [
        'immigration reform',  # Dem focus
        'immigration policy',
        'border security',  # Both parties
        'border',  # Rep focus
        'southern border',  # Rep focus
        'border crisis',  # Rep focus
        'illegal immigration',  # Rep focus
        'immigration enforcement',  # Rep focus
        'secure the border',  # Rep focus
        'border wall',  # Rep focus
        'sanctuary cities',  # Rep focus
        'illegal',  # Rep focus
        'amnesty',  # Rep focus
    ],
    
    'Social Security and Medicare': [
        'social security',  # Both parties
        'medicare',  # Note: also in Healthcare - intentional
        'medicaid',  # Note: also in Healthcare - intentional
        'child care',  # Dem focus
        'benefit',  # Dem focus
        'assistance',  # Dem focus
        'entitlements',  # Rep frame
        'welfare',  # Rep frame
    ],
    
    'Congress & Legislation': [
        'legislation',
        'this bill',
        'the bill',
        'house',
        'senate',
        'congress',
        'committee',
        'law',
        'amendment',
        'veto',
        'majority',
        'majority leader',
    ],
    
    # Democratic-Leaning Categories
    'Voting Rights': [
        'voting rights',
        'voting rights act',
        'election security',  # Note: also in Rep "Election Integrity"
        'vote',
        'voting',
        'election',
    ],
    
    # for presentation
    'Civil Rights': [
        'civil rights',
        "women's rights",
        'equality',
        'equal',
        'discrimination',
        'racial justice',
        'racial inequality',
        'systemic racism',
        'social justice',
        'lgbtq',
        'transgender',
        'trans rights',
        'affirmative action',        
    ],
    
    # for presentation
    'Reproductive Rights': [
        'reproductive rights',
        'abortion rights',
        'abortion access',
        'abortion care',
        'planned parenthood',
        'pro-life',
        'prolife',
        'unborn',
        'sanctity of life',
        'abortion',
        'family planning',
        'birth control',
        'roe v wade',
        'anti-abortion',
    ],
    
    'Climate and Environmental Policy': [
        'climate change',
        'renewable energy',
        'clean energy',
        'green energy',
        'climate',
        'national park',
        'epa',
        'environment',
    ],
    
    # for presentation
    'Criminal Justice Reform': [
        'criminal justice',
        'police reform',
        'criminal justice reform',
        'police',
        'defund the police',
        'back the blue',
        'mass incarceration',
        'prison reform',
        'sentencing reform',
        'police training',
        'community policing',
        'george floyd',
        'breonna taylor',
        'ahmaud arbery',
        'suppport our police',
        'support our law enforcement',
    ],
    
    'Housing & Community': [
        'housing',
        'affordable',
        'community',
        'neighborhood',
    ],
    
    # Republican-Leaning Categories
    'Election Integrity': [
        'election integrity',
        'voter fraud',
        'voter id',
        'election reform',
        'voter verification',
    ],
    
    'Law Enforcement & Crime': [
        'law enforcement',
        'police',
        'law and order',
        'crime',
        'order',
        'public safety',
    ],
    
    'National Security & Defense': [
        'national security',
        'national defense',
        'homeland security',
        'defense',
        'military',
        'veterans',
        'armed forces',
        'defense spending',
    ],
    
    'Terrorism & Foreign Threats': [
        'terrorism',
        'terrorist',
        'terrorists',
        'isis',
        'terror',
        'guantanamo',
    ],
    
    # for presentation
    'Iran & Nuclear Policy': [
        'iran',
        'iran deal',
        'nuclear',
        'sanctions',
        'nuclear deal',
    ],
    
    'Economy & Taxes': [
        'taxes',
        'tax',
        'tax cuts',
        'tax relief',
        'tax reform',
        'tax code',
        'economy',
        'jobs',
        'economic growth',
        'small business',
        'business',
        'regulations',
        'regulatory',
        'regulation',
        'government spending',
        'spending',
        'budget',
        'deficit',
        'middle class',
        'minimum wage',  # Dem focus
        'wage',  # Dem focus
    ],
    
    'Energy Policy': [
        'energy',
        'energy independence',
        'energy production',
        'oil and gas',
        'fossil fuels',
        'pipeline',
        'epa',
        'coal',
        'natural gas',
        'oil',
        'gas',
    ],
    
    'Social Issues & Values': [
        'religious freedom',
        'religious liberty',
        'pro-life',
        'unborn',
        'sanctity of life',
        'family values',
        'traditional values',
        'planned parenthood',
        'abortion',  # Note: different frame than Dem "abortion rights"
        'god',
        'god bless',
        'bless',
        'christmas',
    ],
    
    'Government Overreach': [
        'big government',
        'government overreach',
        'overreach',
        'bureaucracy',
        'bureaucrats',
        'federal government',
        'government',
        'mandates',
        'federal overreach',
        'states rights',
        'states',
        'freedom',
        'liberty',
        'conservative',
        'constitution',
        'constitutional',
        'swamp',
        'drain the swamp',
    ],
    
    'Supreme Court & Judiciary': [
        'supreme court',
        'judicial',
        'judges',
        'department of justice',
        'courts',
    ],
    
    'Executive Power': [
        'executive',
        'executive order',
        'power',
        'accountability',
        'accountable',
        'oversight',
        'transparency',
    ],
    
    'Patriotic Rhetoric': [
        'american',
        'american people',
        'america first'
    ]
}

In [77]:
content = (
    df['Body']
    .apply(remove_double_spaces)
    .apply(remove_paragraphs)
    .apply(remove_paragraphs_dash)
    .to_numpy()
)

In [78]:
content.shape

(20582,)

In [79]:
model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode(content, show_progress_bar=True)

Batches: 100%|██████████| 644/644 [02:26<00:00,  4.38it/s]


In [80]:
print(f"content.shape: {content.shape}")
print(f"X.shape: {X.shape}")


content.shape: (20582,)
X.shape: (20582, 384)


In [81]:
reproductive_rights_terms = THEMATIC_LEXICON['Reproductive Rights']
reproductive_rights_qv = model.encode(reproductive_rights_terms, show_progress_bar=True)

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.87it/s]


In [82]:
rr_similarities = cosine_similarity(reproductive_rights_qv, X)

df_similarities = pd.DataFrame(
    data=rr_similarities.T,  # Transpose so rows=courses, columns=AI keywords
    columns=reproductive_rights_terms
)

df_similarities['ID'] = df['ID'].values


In [83]:
df_similarities.head()

# get top 5 most similar emails for each reproductive rights term
top_n = 5
for term in reproductive_rights_terms:
    top_indices = df_similarities.nlargest(top_n, term).index
    print(f"\nTop {top_n} emails for term '{term}':")
    for idx in top_indices:
        email_id = df_similarities.at[idx, 'ID']
        similarity_score = df_similarities.at[idx, term]
        print(f"Email ID: {email_id}, Similarity Score: {similarity_score:.4f}")


Top 5 emails for term 'reproductive rights':
Email ID: 109730, Similarity Score: 0.5020
Email ID: 111886.0, Similarity Score: 0.5010
Email ID: 116134, Similarity Score: 0.4860
Email ID: 106193, Similarity Score: 0.4854
Email ID: 111362, Similarity Score: 0.4828

Top 5 emails for term 'abortion rights':
Email ID: 96873, Similarity Score: 0.6235
Email ID: 111362, Similarity Score: 0.6229
Email ID: 109233, Similarity Score: 0.6145
Email ID: 106193, Similarity Score: 0.5963
Email ID: 113351, Similarity Score: 0.5917

Top 5 emails for term 'abortion access':
Email ID: 116198, Similarity Score: 0.6265
Email ID: 111362, Similarity Score: 0.6199
Email ID: 115978, Similarity Score: 0.6061
Email ID: 116281, Similarity Score: 0.5977
Email ID: 118103, Similarity Score: 0.5954

Top 5 emails for term 'abortion care':
Email ID: 111362, Similarity Score: 0.6246
Email ID: 116198, Similarity Score: 0.6156
Email ID: 116281, Similarity Score: 0.6117
Email ID: 96873, Similarity Score: 0.6105
Email ID: 115

In [84]:
test = df[df['ID'] == 111910]
test['Body']


12844     Standing up for Life Standing up for life 1/22 Earlier this month, Congressman Doug Lamborn spoke on the Floor of the House in support of the March for Life event taking place today, the 43 rd Anniversary of the tragic Roe v. Wade decision. Please click HERE to watch the Congressman's Speech Text of the Speech Mr. Speaker, it is with a heavy heart that I rise today to speak for those whose lives have been tragically cut short in the wake of Roe v. Wade. A staggering 57 million innocent girls and boys have been legally aborted in this country since that horrible decision 43 years ago. Roe v. Wade remains one of the most heinous acts of judicial activism in the history of the United States. As a father of 5, and a grandfather of 3, I know that every child is a wonderful gift from God. Our country was founded upon the sacred truth that all lives are created equal, and all deserve the right to life, liberty and the pursuit of happiness. The perverse logic that somehow the life of

In [85]:
civil_rights_terms = THEMATIC_LEXICON['Civil Rights']
civil_rights_qv = model.encode(civil_rights_terms, show_progress_bar=True)

Batches: 100%|██████████| 1/1 [00:00<00:00, 20.08it/s]


In [86]:
civil_similarities = cosine_similarity(civil_rights_qv, X)
civil_similarities_df = pd.DataFrame(
    data=civil_similarities.T,  # Transpose so rows=courses, columns=AI keywords
    columns=civil_rights_terms
)
civil_similarities_df['ID'] = df['ID'].values

In [87]:
top_n = 5
for term in civil_rights_terms:
    top_indices = civil_similarities_df.nlargest(top_n, term).index
    print(f"\nTop {top_n} emails for term '{term}':")
    for idx in top_indices:
        email_id = civil_similarities_df.at[idx, 'ID']
        similarity_score = civil_similarities_df.at[idx, term]
        print(f"Email ID: {email_id}, Similarity Score: {similarity_score:.4f}")


Top 5 emails for term 'civil rights':
Email ID: 117975, Similarity Score: 0.5369
Email ID: 112012.0, Similarity Score: 0.5356
Email ID: 109565, Similarity Score: 0.5128
Email ID: 99522, Similarity Score: 0.4955
Email ID: 99354, Similarity Score: 0.4914

Top 5 emails for term 'women's rights':
Email ID: 116961, Similarity Score: 0.5756
Email ID: 98113, Similarity Score: 0.5612
Email ID: 116684, Similarity Score: 0.5427
Email ID: 111300, Similarity Score: 0.5041
Email ID: 121839, Similarity Score: 0.5012

Top 5 emails for term 'equality':
Email ID: 97757, Similarity Score: 0.4234
Email ID: 117975, Similarity Score: 0.3945
Email ID: 116782, Similarity Score: 0.3627
Email ID: 106845, Similarity Score: 0.3549
Email ID: 109093, Similarity Score: 0.3503

Top 5 emails for term 'equal':
Email ID: 97757, Similarity Score: 0.3459
Email ID: 105683, Similarity Score: 0.3402
Email ID: 117975, Similarity Score: 0.3190
Email ID: 97857, Similarity Score: 0.2894
Email ID: 106845, Similarity Score: 0.28

In [88]:
test = df[df['ID'] == '117975']
test['Body']

6792     Click here to open this e-mail in its own browser window Click here to open a plain text version of this email News from Representative Diana DeGette Home | About Diana | News | Contact THIS WEEK IN REVIEW Working for a Long-term Solution to Fix our Crumbling Roads Serving Senior and Connecting Them With Resources Dear Friend, This week, I was proud to join 157 of my colleagues, including civil rights icon, Rep. John Lewis, to introduce the Equality Act, a landmark bill that extends crucial non-discrimination protections to LGBT individuals. The Equality Act amends the Civil Rights Act of 1964 and other existing laws to prohibit discrimination based on sexual orientation or gender identity in the areas of employment, education, credit, housing, federal funding, jury service, and public accommodations. Our society is built on the principle that all are created equal. Discrimination of any kind is not only harmful to each one of us, but at its most basic level, harmful to our de

In [89]:
iran_policy = THEMATIC_LEXICON['Iran & Nuclear Policy']
iran_policy_qv = model.encode(iran_policy, show_progress_bar=True)

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.17it/s]


In [90]:
iran_policy_similarities = cosine_similarity(iran_policy_qv, X)
iran_policy_similarities_df = pd.DataFrame(
    data=iran_policy_similarities.T,  # Transpose so rows=courses, columns=AI keywords
    columns=iran_policy
)
iran_policy_similarities_df['ID'] = df['ID'].values

In [91]:
criminal_justice_terms = THEMATIC_LEXICON['Criminal Justice Reform']
criminal_justice_qv = model.encode(criminal_justice_terms, show_progress_bar=True)

Batches: 100%|██████████| 1/1 [00:00<00:00, 19.71it/s]


In [92]:
criminal_just_similarities = cosine_similarity(criminal_justice_qv, X)
criminal_just_similarities_df = pd.DataFrame(
    data=criminal_just_similarities.T,  # Transpose so rows=courses, columns=AI keywords
    columns=criminal_justice_terms
)
criminal_just_similarities_df['ID'] = df['ID'].values

In [93]:
criminal_just_similarities_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20582 entries, 0 to 20581
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   criminal justice             20582 non-null  float32
 1   police reform                20582 non-null  float32
 2   criminal justice reform      20582 non-null  float32
 3   police                       20582 non-null  float32
 4   defund the police            20582 non-null  float32
 5   back the blue                20582 non-null  float32
 6   mass incarceration           20582 non-null  float32
 7   prison reform                20582 non-null  float32
 8   sentencing reform            20582 non-null  float32
 9   police training              20582 non-null  float32
 10  community policing           20582 non-null  float32
 11  george floyd                 20582 non-null  float32
 12  breonna taylor               20582 non-null  float32
 13  ahmaud arbery   

In [94]:

from functools import reduce

# Put all your similarity dataframes in a list
similarity_dfs = [
    criminal_just_similarities_df,
    civil_similarities_df,
    criminal_just_similarities_df,
    df_similarities,
    # ... add all others
]

# Merge them all on 'ID'
df_all_similarities = reduce(
    lambda left, right: left.merge(right, on='ID', how='outer'),
    similarity_dfs
)

In [96]:
df_all_similarities.info()

# merge df_all_similarities with original df
df_merged = df.merge(df_all_similarities, on='ID', how='left')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20582 entries, 0 to 20581
Data columns (total 60 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   criminal justice_x             20582 non-null  float32
 1   police reform_x                20582 non-null  float32
 2   criminal justice reform_x      20582 non-null  float32
 3   police_x                       20582 non-null  float32
 4   defund the police_x            20582 non-null  float32
 5   back the blue_x                20582 non-null  float32
 6   mass incarceration_x           20582 non-null  float32
 7   prison reform_x                20582 non-null  float32
 8   sentencing reform_x            20582 non-null  float32
 9   police training_x              20582 non-null  float32
 10  community policing_x           20582 non-null  float32
 11  george floyd_x                 20582 non-null  float32
 12  breonna taylor_x               20582 non-null 

In [102]:
email_themes = pd.read_csv('../data/dcInbox/email_themes_dcinbox_114.csv')
email_themes.info()
# get matched_phrases and themes from email_themes and merge into df_merged
email_themes_subset = email_themes[['email_id', 'matched_phrases', 'themes']]
df_final = df_merged.merge(email_themes_subset, left_on='ID', right_on='email_id', how='left')
df_final.info()
df_final.to_csv('../data/dcInbox/dcinbox_114_with_thematic_similarities_from_embeddings.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20462 entries, 0 to 20461
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   first_name       20462 non-null  object 
 1   last_name        20462 non-null  object 
 2   party            20462 non-null  object 
 3   email_id         20462 non-null  float64
 4   subject          20461 non-null  object 
 5   matched_phrases  20462 non-null  object 
 6   themes           20462 non-null  object 
 7   email            20462 non-null  object 
dtypes: float64(1), object(7)
memory usage: 1.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20582 entries, 0 to 20581
Data columns (total 78 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Subject                        20581 non-null  object        
 1   Body                           20582 non-null  object        
 2 